<a href="https://colab.research.google.com/github/SruthiGS-Gito/Online-News-Popularity-Prediction/blob/feature%2Fknn-regression/Online_News_Popularity_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Libraries

In [1]:
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score, accuracy_score
from sklearn.naive_bayes import GaussianNB

# Reading Dataset

In [ ]:
# filepath to the csv file
filepath = '/content/drive/MyDrive/Colab Notebooks/OnlineNewsPopularity.csv'

# Reading dataset and assigning the df created to a variable so as to store it
df_news = pd.read_csv(filepath)
df_news.columns = df_news.columns.str.strip()
df_news.head()

# EDA

In [ ]:
df_news.shape

In [ ]:
df_news.info()

In [ ]:
df_news.describe()

In [ ]:
df_news.isnull().sum()

In [ ]:
df_news.duplicated().sum()

In [ ]:
# "url" is an article link and "timedelta" is days since published, both doesn't predict news popularity hence Drop
df_news.drop(columns = ["url", "timedelta"], inplace = True)
df_news.head()

In [ ]:
num_cols = []
cat_cols = []

In [ ]:
num_cols = df_news.select_dtypes(include = ['int','float']).columns
num_cols

In [ ]:
cat_cols = df_news.select_dtypes(include = ['object']).columns
cat_cols    # empty hence no encoding required

### Correlation & Mutual Information

In [ ]:
# correlation gives linear relationships with "shares"
corr_with_target = df_news.corr()['shares'].sort_values(ascending=False)
corr_with_target

In [ ]:
X_temp = df_news.drop(columns = ["shares"])
y_temp = df_news["shares"]

# mutual_info_regression is used because our target column "shares" has numeric/continous values
mi_scores = mutual_info_regression(X_temp, y_temp, random_state = 42)
# for "number to its features mapping" we use pd.Series()
mi_series = pd.Series(mi_scores, index = X_temp.columns).sort_values(ascending = False)
mi_series

### Plots

In [ ]:
# selecting key columns to analyze
key_cols = mi_series.head(10).index.tolist()
key_cols

In [ ]:
# plotting distribution of each key column using histograms
plt.figure(figsize=(14,10))
for i, col in enumerate(key_cols):
    plt.subplot(5, 2, i+1)
    sns.histplot(df_news[col]) # shows spread/skeweness of each variable
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
# plotting boxplots to check for outliers in each column
plt.figure(figsize=(14,10))
for i, col in enumerate(key_cols):
    plt.subplot(5, 2, i+1)
    sns.boxplot(df_news[col].dropna())   # highlights outliers beyond whiskers
    plt.title(col)

plt.tight_layout()
plt.show()

# PreProcessing

## Data cleaning

### Missing Value Handling

In [ ]:
# There is no missing values to be handled.

### Outlier Handling

In [ ]:
out_cols = [col for col in key_cols if col != 'shares']
out_cols

In [ ]:
for col in out_cols:
    Q1 = df_news[col].quantile(0.25)
    Q3 = df_news[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_news[col] = np.where(df_news[col] < lower, lower, df_news[col])
    df_news[col] = np.where(df_news[col] > upper, upper, df_news[col])

### Duplicates Removal

In [ ]:
# There is no duplicates to remove

### Data Split

In [ ]:
# split before scaling to prevent data leakage
X = df_news.drop(columns = ["shares"])
y = df_news["shares"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Data Transformation

### Scaling

In [ ]:
# StandardScaler() uses mean as 0 and std as 1, Data is skewed with outliers and since min-max is sensitive to outliers we use StandardScaler()
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)     # fits only on train data
X_test = scaler.transform(X_test)       # transforms test using train's data

### Encoding

In [ ]:
# No categorical columns after dropping 'url' hence nothing to encode

# Model Building

In [ ]:
# converting shares into a binary classification target using median as threshold
median_shares = df_news['shares'].median()
df_news['popular'] = (df_news['shares'] > median_shares).astype(int)

# Define X and y for classification using new variable names
y_classification = df_news['popular']
X_classification = df_news.drop(columns=['shares', 'popular'])

# Perform train-test split for classification using new variable names
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_classification, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

# Apply StandardScaler to the classification-ready data using new variable names
scaler_clf = StandardScaler()
X_train_clf = scaler_clf.fit_transform(X_train_clf)
X_test_clf = scaler_clf.transform(X_test_clf)

## KNN

In [ ]:
# build the model object
knn_obj = KNeighborsRegressor(n_neighbors = 4)
# Train the model
knn_obj.fit(X_train, y_train)

# make predictions using the model
y_pred_knn = knn_obj.predict(X_test)

# evaluate the model performance with regression metrics
knn_r2 = r2_score(y_test, y_pred_knn)
knn_rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

df_mae = mean_absolute_error(y_test, y_pred_knn)
df_mse = mean_squared_error(y_test, y_pred_knn)
df_rmse = np.sqrt(df_mse)

print("R2 Score : ", knn_r2)
print("RMSE : ", knn_rmse)
print("Mean absolute error :",df_mae)
print("Mean squared error :",df_mse)
print("Root Mean squared error :",df_rmse)


## Cross Validation

### K-Fold

In [ ]:
kf_obj = KFold(n_splits = 5, shuffle = True,)
scores = cross_val_score(KNeighborsRegressor(), X, y, cv = kf_obj)
print('For Decision Tree model')
print("cross validation scores:",-scores)
print("mean of cv scores:",-scores.mean())

print('***************************')

### Grid search

In [ ]:
# Define the parameter grid for KNeighborsRegressor
param_grid_knn = {
    'n_neighbors': [5, 10, 15, 20, 30, 50],   # Number of neighbors to consider
    'weights': ['uniform', 'distance'],       # Whether closer neighbors count more
    'p': [1, 2]                               # 1 = Manhattan distance, 2 = Euclidean distance
}
# Initialize GridSearchCV
grid_search_knn = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param_grid_knn,
    cv=kf_obj, # Using the kf_obj defined earlier for consistency
    scoring='neg_mean_absolute_error', # Optimize for MAE
    n_jobs=-1, # Use all available cores
    verbose=1
)
# Fit GridSearchCV to the training data
grid_search_knn.fit(X_train, y_train)
# Print the best parameters and best MAE score
print("Best parameters for KNN Regressor (Grid Search):", grid_search_knn.best_params_)
print("Best MAE (Grid Search):", -grid_search_knn.best_score_)

In [ ]:
# Get the best model from Grid Search
best_knn_model = grid_search_knn.best_estimator_
# Make predictions on the test set
y_pred_knn_modf = best_knn_model.predict(X_test)
# Evaluate the tuned model
knn_mae_modf = mean_absolute_error(y_test, y_pred_knn_modf)
knn_mse_modf = mean_squared_error(y_test, y_pred_knn_modf)
knn_rmse_modf = root_mean_squared_error(y_test, y_pred_knn_modf)
print("Mean absolute error : ", knn_mae_modf)
print("Mean squared error : ", knn_mse_modf)
print("Root mean squared error : ", knn_rmse_modf)

### Random Search

In [ ]:
kf_obj = KFold(n_splits=5, shuffle=True, random_state=42) # Define kf_obj to prevent NameError

# Define the parameter distribution for KNN Regressor
param_dist_knn = {
    'n_neighbors': randint(1, 31),      # Number of neighbors from 1 to 30
    'weights': ['uniform', 'distance'], # Weight function
    'p': [1, 2]                         # 1 = Manhattan, 2 = Euclidean distance
}

# Initialize RandomizedSearchCV
random_search_knn = RandomizedSearchCV(
    estimator=KNeighborsRegressor(),
    param_distributions=param_dist_knn,
    n_iter=50,                          # Number of parameter settings sampled
    cv=kf_obj,                          # Using the previously defined K-Fold object
    scoring='neg_mean_absolute_error',  # Optimize for MAE
    random_state=42,
    n_jobs=-1,                          # Use all available CPU cores
    verbose=1
)

# Fit RandomizedSearchCV to the training data
random_search_knn.fit(X_train, y_train) # Use regression data

# Print the best parameters and best MAE score
print("Best parameters for KNN Regressor (Random Search):", random_search_knn.best_params_)
print("Best MAE (Random Search):", -random_search_knn.best_score_)

# Ensemble model

## Bagging

In [ ]:
# build model object
bagging_model = BaggingRegressor(estimator = KNeighborsRegressor(),
                                  n_estimators = 10,
                                  max_samples = 0.6,
                                  random_state=42)
# train model
bagging_model.fit(X_train, y_train)
# make predictions using the model
y_pred_bag = bagging_model.predict(X_test)
# evaluate model performance
bagging_mae = mean_absolute_error(y_test, y_pred_bag)
bagging_mse = mean_squared_error(y_test, y_pred_bag)
bagging_rmse = root_mean_squared_error(y_test, y_pred_bag)

print("Mean absolute error (Bagging) : ", bagging_mae)
print("Mean squared error (Bagging) : ", bagging_mse)
print("Root mean squared error (Bagging) : ", bagging_rmse)

## Boosting

In [ ]:
# Build the model
knn_model = KNeighborsRegressor(n_neighbors=5)

# Train
knn_model.fit(X_train, y_train)

# Predict
knn_pred = knn_model.predict(X_test)

# Evaluate
knn_mae = mean_absolute_error(y_test, knn_pred)
knn_mse = mean_squared_error(y_test, knn_pred)
knn_rmse = root_mean_squared_error(y_test, knn_pred)

print("Mean Absolute Error (KNN):", knn_mae)
print("Mean Squared Error (KNN):", knn_mse)
print("Root Mean Squared Error (KNN):", knn_rmse)